In [1]:
import os
import time
import sys
from pathlib import Path

import json
import uuid

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import col, concat_ws, current_timestamp, lit, sha2
from pyspark.sql.types import (
    LongType,
    StringType,
    StructField,
    StructType,
)

from etl.transformations.common import (
    create_spark,
    read_clickhouse,
    read_current_snapshot,
    write_clickhouse,
)

from etl.transformations.dimensions.scd.common import (
    build_expired_version,
    build_new_version,
    add_hash,
    split_changes,
)

MINIO_STAGING_BUCKET = os.environ.get(
    "MINIO_STAGING_BUCKET",
    "staging",
)

In [2]:
def transform_dim_sensor_type(
    sensor_type_df: DataFrame,
) -> DataFrame:
    """
    Build dim_sensor_type warehouse shape from source snapshot.
    """

    return sensor_type_df.select(
        sensor_type_df.id.alias("sensor_type_id"),
        sensor_type_df.name,
        sensor_type_df.unit,
        sensor_type_df.description,
        sensor_type_df.optimal_min,
        sensor_type_df.optimal_max,
    )

In [3]:
def main():
    """
    Load dim_sensor_type as an SCD2 dimension.
    """

    spark = create_spark(
        "load_dim_sensor_type",
    )

    try:
        #
        # 1. Read current source snapshot
        #
        sensor_type_df = read_current_snapshot(
            spark,
            MINIO_STAGING_BUCKET,
            "sensor_types",
        )

        #
        # 2. Transform source shape
        #
        dim_sensor_type_source_df = transform_dim_sensor_type(
            sensor_type_df,
        )

        #
        # 3. Read current active SCD2 rows
        #
        current_dim_sensor_type_df = read_clickhouse(
            spark,
            """
            (
                SELECT *
                FROM dim_sensor_type FINAL
            ) AS dim_sensor_type
            """,
        ).filter(
            col("is_current") == 1,
        )

        #
        # 4. Hash attributes for change detection
        #
        source_hashed_df = add_hash(
            dim_sensor_type_source_df,
            [
                "name",
                "unit",
                "description",
                "optimal_min",
                "optimal_max",
            ],
        )

        current_hashed_df = add_hash(
            current_dim_sensor_type_df,
            [
                "name",
                "unit",
                "description",
                "optimal_min",
                "optimal_max",
            ],
        )

        #
        # 5. Find new and changed rows
        #
        new_sensor_types_df, changed_sensor_types_df = split_changes(
            source_hashed_df,
            current_hashed_df,
            "sensor_type_id",
        )

        #
        # 6. Find existing versions that need closing
        #
        expired_sensor_types_df = current_dim_sensor_type_df.join(
            changed_sensor_types_df.select(
                "sensor_type_id",
            ),
            "sensor_type_id",
            "inner",
        )

        #
        # 7. Create SCD2 versions
        #
        load_version = int(
            time.time() * 1000,
        )

        build_new_version(
                new_sensor_types_df,
                load_version,
                ["sensor_type_key"],
            ).show()

        build_expired_version(
                    expired_sensor_types_df,
                    load_version,
                    ["sensor_type_key"],
                )

        rows_to_write = (
            build_new_version(
                new_sensor_types_df,
                load_version,
                ["sensor_type_key"],
            )
            .unionByName(
                build_expired_version(
                    expired_sensor_types_df,
                    load_version,
                    ["sensor_type_key"],
                )
            )
            .unionByName(
                build_new_version(
                    changed_sensor_types_df,
                    load_version,
                    ["sensor_type_key"],
                )
            )
        )

        #
        # 8. Write to ClickHouse
        #
        write_clickhouse(
            rows_to_write,
            "dim_sensor_type",
        )

    finally:
        spark.stop()

In [4]:
if __name__ == "__main__":
    main()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/22 14:35:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/22 14:35:54 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+--------------+-----------------+----+--------------------+-----------+-----------+--------------------+-------------------+----------+-------------+
|sensor_type_id|             name|unit|         description|optimal_min|optimal_max|          valid_from|           valid_to|is_current|     _version|
+--------------+-----------------+----+--------------------+-----------+-----------+--------------------+-------------------+----------+-------------+
|             1|      Temperature|  °C|Ambient temperatu...|     18.000|     25.000|2026-07-22 14:35:...|2099-12-31 23:59:59|         1|1784730957188|
|             2|         Humidity|   %|Relative humidity...|     50.000|     70.000|2026-07-22 14:35:...|2099-12-31 23:59:59|         1|1784730957188|
|             3|  Light Intensity|PPFD|Photosynthetic ph...|    200.000|    800.000|2026-07-22 14:35:...|2099-12-31 23:59:59|         1|1784730957188|
|             4|         pH Level|  pH|Acidity/alkalinit...|      5.000|      7.000|2026-07-22

26/07/22 14:36:00 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
